# Google FRAMES Benchmark — EDA & Preprocessing

**FRAMES** (Factuality, Retrieval, And Multi-hop Evaluation of RAG Systems) is a benchmark for evaluating RAG pipelines on multi-hop questions that require reasoning over multiple Wikipedia articles.

This notebook covers:
1. Loading the dataset
2. Basic structure & schema inspection
3. Exploratory Data Analysis (distributions, lengths, label breakdown)
4. Preprocessing (cleaning, tokenisation-ready features)
5. Export to a clean Parquet file

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_colwidth', 120)

print('All imports OK')

## 1. Load the Dataset

In [ ]:
ds = load_dataset('google/frames-benchmark')
print(ds)

In [ ]:
# Convert every split to a DataFrame and keep track of them
splits = {split: ds[split].to_pandas() for split in ds.keys()}

for name, df in splits.items():
    print(f"Split '{name}': {len(df):,} rows  |  columns: {list(df.columns)}")

## 2. Schema Inspection

In [ ]:
# Use the largest split (test) for EDA; fall back to the first available split
primary_split = 'test' if 'test' in splits else list(splits.keys())[0]
df = splits[primary_split].copy()
print(f"Working with split: '{primary_split}'  ({len(df):,} rows)")
df.head(3)

In [ ]:
df.dtypes

In [ ]:
print("=== Missing values ===")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## 3. Exploratory Data Analysis

### 3.1 Reasoning Type Distribution

In [ ]:
reasoning_col = None
for candidate in ['reasoning_types', 'reasoning_type', 'type', 'category']:
    if candidate in df.columns:
        reasoning_col = candidate
        break

if reasoning_col:
    # Column may contain lists (multi-label) or strings
    if df[reasoning_col].apply(lambda x: isinstance(x, list)).any():
        from itertools import chain
        all_types = list(chain.from_iterable(df[reasoning_col].dropna()))
    else:
        all_types = df[reasoning_col].dropna().tolist()

    type_counts = pd.Series(all_types).value_counts()

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(type_counts.index, type_counts.values, color=sns.color_palette('muted', len(type_counts)))
    ax.bar_label(bars, padding=4, fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title('Reasoning Type Distribution')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print(type_counts.to_string())
else:
    print(f"No reasoning-type column found. Available: {list(df.columns)}")

### 3.2 Question Length Distribution

In [ ]:
question_col = None
for candidate in ['Prompt', 'prompt', 'question', 'query', 'input']:
    if candidate in df.columns:
        question_col = candidate
        break

if question_col:
    df['q_char_len'] = df[question_col].str.len()
    df['q_word_len'] = df[question_col].str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, label in zip(axes,
                               ['q_char_len', 'q_word_len'],
                               ['Character length', 'Word count']):
        ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
        ax.set_xlabel(label)
        ax.set_ylabel('Frequency')
        ax.set_title(f'Question {label} — {primary_split}')
    plt.tight_layout()
    plt.show()

    print(df[['q_char_len', 'q_word_len']].describe().round(1))
else:
    print(f'Question column not found. Columns: {list(df.columns)}')

### 3.3 Answer Length Distribution

In [ ]:
answer_col = None
for candidate in ['Answer', 'answer', 'label', 'output', 'response']:
    if candidate in df.columns:
        answer_col = candidate
        break

if answer_col:
    df['a_char_len'] = df[answer_col].astype(str).str.len()
    df['a_word_len'] = df[answer_col].astype(str).str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, label in zip(axes,
                               ['a_char_len', 'a_word_len'],
                               ['Character length', 'Word count']):
        ax.hist(df[col].dropna(), bins=40, color='coral', edgecolor='white')
        ax.set_xlabel(label)
        ax.set_ylabel('Frequency')
        ax.set_title(f'Answer {label} — {primary_split}')
    plt.tight_layout()
    plt.show()

    print(df[['a_char_len', 'a_word_len']].describe().round(1))
else:
    print(f'Answer column not found. Columns: {list(df.columns)}')

### 3.4 Number of Wikipedia Links per Example

In [ ]:
wiki_col = None
for candidate in ['wiki_links', 'links', 'passages', 'context', 'documents', 'urls']:
    if candidate in df.columns:
        wiki_col = candidate
        break

if wiki_col:
    df['n_links'] = df[wiki_col].apply(
        lambda x: len(x) if isinstance(x, list) else (1 if pd.notna(x) else 0)
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    counts = df['n_links'].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='mediumseagreen', edgecolor='white')
    ax.set_xlabel('Number of wiki links')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Wiki Links per Example')
    plt.tight_layout()
    plt.show()
    print(df['n_links'].describe().round(2))
else:
    # Try to detect a URL-like column automatically
    url_cols = [c for c in df.columns if 'url' in c.lower() or 'link' in c.lower() or 'wiki' in c.lower()]
    print(f'No wiki-link column found. Possible candidates: {url_cols}')
    print(f'All columns: {list(df.columns)}')

### 3.5 Question vs Answer Length Scatter

In [ ]:
if question_col and answer_col:
    fig, ax = plt.subplots(figsize=(7, 5))
    hue_col = reasoning_col if reasoning_col else None

    if hue_col and not df[hue_col].apply(lambda x: isinstance(x, list)).any():
        for rtype, grp in df.groupby(hue_col):
            ax.scatter(grp['q_word_len'], grp['a_word_len'], alpha=0.5, label=rtype, s=20)
        ax.legend(title='Reasoning type', fontsize=8, markerscale=1.5)
    else:
        ax.scatter(df['q_word_len'], df['a_word_len'], alpha=0.4, s=20, color='slateblue')

    ax.set_xlabel('Question word count')
    ax.set_ylabel('Answer word count')
    ax.set_title('Question vs Answer Length')
    plt.tight_layout()
    plt.show()
else:
    print('Need both question and answer columns for this plot.')

## 4. Preprocessing

### 4.1 Normalise Text Fields

In [ ]:
import re

def clean_text(text: str) -> str:
    """Lowercase, collapse whitespace, strip leading/trailing spaces."""
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)  # remove control characters
    return text

text_cols = [c for c in df.columns if df[c].dtype == object]
print(f'Text columns to clean: {text_cols}')

df_clean = df.copy()
for col in text_cols:
    # Only apply to columns that are purely string (not list)
    if df_clean[col].apply(lambda x: isinstance(x, str) or pd.isna(x)).all():
        df_clean[col] = df_clean[col].apply(clean_text)

print('\nText cleaning done.')

### 4.2 Drop / Fill Missing Values

In [ ]:
before = len(df_clean)

# Drop rows where the question or answer is empty
if question_col:
    df_clean = df_clean[df_clean[question_col].str.len() > 0]
if answer_col:
    df_clean = df_clean[df_clean[answer_col].str.len() > 0]

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f'Rows before: {before:,}  |  after cleaning: {len(df_clean):,}  |  dropped: {before - len(df_clean)}')

### 4.3 Encode Reasoning Types (One-Hot)

In [ ]:
if reasoning_col:
    is_list_col = df_clean[reasoning_col].apply(lambda x: isinstance(x, list)).any()

    if is_list_col:
        # Multi-label one-hot
        mlb_df = df_clean[reasoning_col].apply(
            lambda x: x if isinstance(x, list) else []
        )
        ohe = mlb_df.explode().str.strip().str.lower()
        ohe_dummies = pd.get_dummies(ohe).groupby(level=0).max()
        ohe_dummies.columns = [f'type_{c}' for c in ohe_dummies.columns]
        df_clean = pd.concat([df_clean, ohe_dummies], axis=1)
        print(f'Added {len(ohe_dummies.columns)} one-hot columns: {list(ohe_dummies.columns)}')
    else:
        # Single-label encoding
        df_clean['reasoning_type_clean'] = df_clean[reasoning_col].str.strip().str.lower()
        df_clean['reasoning_type_id'] = df_clean['reasoning_type_clean'].astype('category').cat.codes
        print(df_clean[['reasoning_type_clean', 'reasoning_type_id']].value_counts())
else:
    print('No reasoning type column — skipping encoding.')

### 4.4 Add Token-Count Estimates

In [ ]:
# Rule-of-thumb: ~4 characters per token (GPT/Claude tokenisers)
CHARS_PER_TOKEN = 4

if question_col:
    df_clean['q_est_tokens'] = (df_clean[question_col].str.len() / CHARS_PER_TOKEN).round().astype(int)
if answer_col:
    df_clean['a_est_tokens'] = (df_clean[answer_col].str.len() / CHARS_PER_TOKEN).round().astype(int)

token_cols = [c for c in ['q_est_tokens', 'a_est_tokens'] if c in df_clean.columns]
if token_cols:
    print(df_clean[token_cols].describe().round(1))

### 4.5 Final Schema Overview

In [ ]:
print(f'Final shape: {df_clean.shape}')
df_clean.dtypes

In [ ]:
df_clean.head(3)

## 5. Export

In [ ]:
out_path = f'frames_{primary_split}_preprocessed.parquet'

# Parquet can't serialise Python list columns natively — convert to string
for col in df_clean.columns:
    if df_clean[col].apply(lambda x: isinstance(x, list)).any():
        df_clean[col] = df_clean[col].astype(str)

df_clean.to_parquet(out_path, index=False)
print(f'Saved → {out_path}  ({len(df_clean):,} rows, {df_clean.shape[1]} columns)')

---
### Summary

| Step | What was done |
|------|---------------|
| Load | `datasets.load_dataset('google/frames-benchmark')` |
| Schema | Checked dtypes, nulls, duplicates |
| EDA | Reasoning type counts, question/answer length distributions, wiki-link counts, scatter plot |
| Clean | Whitespace normalisation, control-char removal, empty-row removal |
| Encode | One-hot (multi-label) or ordinal encoding of reasoning types |
| Features | Estimated token counts for questions and answers |
| Export | Saved to `.parquet` for downstream use |